In [28]:
import numpy as np
import torch
from torch.utils.data import Dataset
from MuseCar_Embedding_generation.utils import load_embeddings, load_labels

In [29]:
video_dir = "D:\c2_muse_sent\Video_output_512_DINOv2"
audio_dir = "D:\c2_muse_sent\Audio_output_512_HuBERT"
text_dir = "D:\c2_muse_sent\Text_output_512_BERT"
labels_file = "C:/Users/aryan/Documents/Study/Research/MuseCar_Classification/c2_muse_topic/label_segments/aggregated/train.csv"
config_file = "my_custom_model\config.json"

In [30]:
video_embeddings = load_embeddings(video_dir)
audio_embeddings = load_embeddings(audio_dir)
text_embeddings = load_embeddings(text_dir)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\c2_muse_sent\\Text_output_512_BERT'

In [31]:
print(audio_embeddings[(1,1)])

[[-0.09419083  0.14525832  0.34800798 ...  0.00208853  0.0389427
   0.08728742]
 [-0.31632122  0.23730679  0.3627835  ...  0.26126435 -0.07512856
  -0.29703388]
 [ 0.19079483 -0.07765353  0.19004834 ...  0.21673128  0.07979801
   0.4448403 ]
 ...
 [ 0.24357577 -0.29432893  0.16694395 ...  0.22695883  0.16361657
   1.320533  ]
 [-0.07341351 -0.14532126  0.1057499  ...  0.36970633  0.15815738
   0.9398582 ]
 [ 0.01485614 -0.10550915 -0.02523395 ... -0.20047186 -0.14901178
   0.02725519]]


In [32]:
print(video_embeddings[(1,1)].shape)

(512, 768)


In [33]:
labels = load_labels("C:/Users/aryan/Documents/Study/Research/MuseCar_Classification/c2_muse_topic/label_segments/aggregated/train.csv")

In [34]:
common_keys = (
    set(video_embeddings.keys())
    & set(audio_embeddings.keys())
    & set(text_embeddings.keys())
    & set(labels.keys())
)

common_keys = sorted(list(common_keys))

AttributeError: 'Tensor' object has no attribute 'keys'

In [35]:
class MultimodalDataset(Dataset):
    def __init__(self, video_embeds, audio_embeds, text_embeds, labels):
        
        self.video_embeds = video_embeds
        self.audio_embeds = audio_embeds
        self.text_embeds = text_embeds
        self.labels = labels

        self.keys = sorted(
            list(
                set(video_embeds.keys())
                & set(audio_embeds.keys())
                & set(text_embeds.keys())
                & set(labels.keys())
            )
        )

        print(f"Dataset size: {len(self.keys)} samples")

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]

        video = torch.tensor(self.video_embeds[key], dtype=torch.float32)
        audio = torch.tensor(self.audio_embeds[key], dtype=torch.float32)
        text  = torch.tensor(self.text_embeds[key], dtype=torch.float32)

        label_info = self.labels[key]

        arousal = torch.tensor(label_info["arousal"], dtype=torch.long)
        valence = torch.tensor(label_info["valence"], dtype=torch.long)
        topic   = torch.tensor(label_info["topic"], dtype=torch.long)

        return {
            "video": video,
            "audio": audio,
            "text": text,
            "arousal": arousal,
            "valence": valence,
            "topic": topic
        }

In [36]:
dataset = MultimodalDataset(
    video_embeddings,
    audio_embeddings,
    text_embeddings,
    labels
)

sample = dataset[0]

print(sample["video"].shape)
print(sample["arousal"])

AttributeError: 'Tensor' object has no attribute 'keys'